In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path


In [2]:
# ------------------------- 1. LOAD DATA ---------------------------

EXCEL_FILE = "EcoMoveMobility.xlsx"

# Automatically look for an Excel filename containing EcoMoveMobility
# in the current working folder if the default name is not found.
if not Path(EXCEL_FILE).exists():
    matches = list(Path(".").glob("*EcoMoveMobility*.xlsx"))
    if matches:
        EXCEL_FILE = str(matches[0])
    else:
        raise FileNotFoundError(
            "Cannot find the EcoMoveMobility Excel file. Put it in the "
            "same folder as this notebook, or set EXCEL_FILE to its full path."
        )

df = pd.read_excel(EXCEL_FILE)

print("Dataset loaded successfully.")
print("Excel file:", EXCEL_FILE)
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))
display(df.head())


Dataset loaded successfully.
Excel file: [5442] EcoMoveMobility.xlsx
Number of rows: 5000
Number of columns: 18


,Record_ID,Date,City,Country,Transport_Mode,Route_Type,Weather,Event_Day,Passengers,Trips,Average_Delay_Minutes,On_Time_Percentage,Ticket_Revenue_EUR,Operating_Cost_EUR,CO2_Saved_KG,Energy_Consumption_KWh,Customer_Rating,Accessibility_Complaints
0,EM-00001,45747,Paris,France,Tram,Tourist Zone,Cloudy,Yes,22946,101,3.2,85.1,58846.34,47796.73,2005.0,1446.4,4.4,26
1,EM-00002,45717,Paris,France,Tram,University Area,Cloudy,No,9417,59,1.5,90.1,24578.85,20708.01,824.3,495.1,4.7,7
2,EM-00003,45882,Rotterdam,Netherlands,Tram,Airport Connector,Clear,No,6467,38,3.4,87.9,15168.68,14129.41,693.1,405.8,4.4,7
3,EM-00004,45504,Lyon,France,Tram,University Area,Clear,Yes,5218,33,6.9,74.5,14172.20,11945.52,600.1,278.1,3.9,7
4,EM-00005,45578,Amsterdam,Netherlands,Metro,Tourist Zone,Cloudy,No,10258,19,3.6,83.2,27534.92,28846.82,1392.4,580.5,4.4,12


In [3]:
# ------------------------- 2. CHECK DATA --------------------------

required_columns = [
    "Record_ID", "Date", "City", "Country", "Transport_Mode",
    "Route_Type", "Weather", "Event_Day", "Passengers", "Trips",
    "Average_Delay_Minutes", "On_Time_Percentage",
    "Ticket_Revenue_EUR", "Operating_Cost_EUR", "CO2_Saved_KG",
    "Energy_Consumption_KWh", "Customer_Rating",
    "Accessibility_Complaints"
]

missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing columns in Excel file: {missing_columns}")

# The supplied Excel file stores Date as Excel serial numbers.
# Convert serial dates using Excel's standard date origin.
if pd.api.types.is_numeric_dtype(df["Date"]):
    df["Date"] = pd.to_datetime(
        df["Date"], unit="D", origin="1899-12-30", errors="coerce"
    )
else:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Convert numeric fields safely.
numeric_columns = [
    "Passengers", "Trips", "Average_Delay_Minutes",
    "On_Time_Percentage", "Ticket_Revenue_EUR",
    "Operating_Cost_EUR", "CO2_Saved_KG",
    "Energy_Consumption_KWh", "Customer_Rating",
    "Accessibility_Complaints"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["Date"]).copy()
df["Month"] = df["Date"].dt.to_period("M").dt.to_timestamp()
df["Net_Operating_Result_EUR"] = (
    df["Ticket_Revenue_EUR"] - df["Operating_Cost_EUR"]
)

if df.empty:
    raise ValueError("No valid dated records remain after date conversion.")

print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing values"))




Data types:
Record_ID                           object
Date                        datetime64[ns]
City                                object
Country                             object
Transport_Mode                      object
Route_Type                          object
Weather                             object
Event_Day                           object
Passengers                           int64
Trips                                int64
Average_Delay_Minutes              float64
On_Time_Percentage                 float64
Ticket_Revenue_EUR                 float64
Operating_Cost_EUR                 float64
CO2_Saved_KG                       float64
Energy_Consumption_KWh             float64
Customer_Rating                    float64
Accessibility_Complaints             int64
Month                       datetime64[ns]
Net_Operating_Result_EUR           float64
dtype: object

Missing values:


,Missing values
Record_ID,0
Date,0
City,0
Country,0
Transport_Mode,0
Route_Type,0
Weather,0
Event_Day,0
Passengers,0
Trips,0


In [4]:
# ------------------------- 3. SUMMARY -----------------------------

print("\nDESCRIPTIVE STATISTICS")
display(df[numeric_columns].describe().round(2))

print("\nSUMMARY BY CITY")
city_summary = (
    df.groupby("City")
      .agg(
          Passengers=("Passengers", "sum"),
          Trips=("Trips", "sum"),
          Average_Delay=("Average_Delay_Minutes", "mean"),
          On_Time_Percentage=("On_Time_Percentage", "mean"),
          Revenue=("Ticket_Revenue_EUR", "sum"),
          Operating_Cost=("Operating_Cost_EUR", "sum"),
          CO2_Saved=("CO2_Saved_KG", "sum"),
          Accessibility_Complaints=("Accessibility_Complaints", "sum")
      )
      .sort_values("Passengers", ascending=False)
      .round(2)
)
display(city_summary)



DESCRIPTIVE STATISTICS


,Passengers,Trips,Average_Delay_Minutes,On_Time_Percentage,Ticket_Revenue_EUR,Operating_Cost_EUR,CO2_Saved_KG,Energy_Consumption_KWh,Customer_Rating,Accessibility_Complaints
count,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00
mean,8181.94,121.77,4.35,82.06,22328.12,22422.00,896.39,466.81,4.01,12.53
std,10018.72,155.79,2.39,7.93,28822.41,26114.32,1261.16,564.22,0.40,16.97
min,64.00,11.00,0.00,55.00,90.93,978.37,8.30,0.00,2.70,0.00
25%,1411.75,36.00,2.60,76.50,3280.33,7189.86,126.85,85.20,3.70,2.00
50%,4398.00,52.00,4.40,82.00,11190.02,11638.28,338.30,269.70,4.00,7.00
75%,10077.50,106.00,6.00,87.90,26329.65,22732.98,1009.48,599.62,4.30,15.00
max,66416.00,1048.00,12.50,99.00,204386.05,178936.39,8884.10,3856.30,4.90,147.00



SUMMARY BY CITY


,Passengers,Trips,Average_Delay,On_Time_Percentage,Revenue,Operating_Cost,CO2_Saved,Accessibility_Complaints
City,,,,,,,,
Paris,4492481,70873,4.36,81.96,12353471.88,12359500.35,488250.7,6901
Berlin,4487351,59444,4.55,81.37,12254517.96,12239974.21,493559.2,7121
Munich,3603328,47450,4.38,82.01,9912290.08,9741051.40,397614.7,5379
Barcelona,3526320,54291,4.11,82.80,9642790.24,9795484.13,389670.0,5402
Rome,3418626,53761,4.16,82.71,9286877.93,9381339.80,374058.8,5186
Vienna,3388470,44462,4.32,82.39,9321041.96,9230946.16,376563.4,4945
Amsterdam,3364412,47964,4.44,81.70,9139564.71,9169223.29,363691.3,5038
Madrid,3289904,57267,4.35,82.00,8893197.14,9165684.00,356978.9,5079
Milan,3100424,49961,4.33,82.20,8421557.29,8567414.92,346878.8,4864


In [5]:
# ------------------------- 4. CHART 1 -----------------------------
# Time-series line chart: passenger demand over time

monthly_data = (
    df.groupby("Month", as_index=False)
      .agg(Passengers=("Passengers", "sum"),
           Trips=("Trips", "sum"))
)

fig1 = px.line(
    monthly_data,
    x="Month",
    y="Passengers",
    markers=True,
    title="1. Monthly Passenger Demand Over Time",
    labels={"Month": "Month", "Passengers": "Total passengers"},
    template="plotly_white"
)
fig1.update_layout(hovermode="x unified")
fig1.show()



In [6]:
# ------------------------- 5. CHART 2 -----------------------------
# Comparison bar chart: passenger demand by city

city_data = (
    df.groupby("City", as_index=False)["Passengers"]
      .sum()
      .sort_values("Passengers", ascending=False)
)

fig2 = px.bar(
    city_data,
    x="City",
    y="Passengers",
    color="Passengers",
    text_auto=".2s",
    title="2. Total Passenger Demand by City",
    labels={"Passengers": "Total passengers"},
    template="plotly_white",
    color_continuous_scale="Blues"
)
fig2.update_layout(xaxis_tickangle=-30, coloraxis_showscale=False)
fig2.show()



In [7]:
# ------------------------- 6. CHART 3 -----------------------------
# Transport mode comparison

mode_data = (
    df.groupby("Transport_Mode", as_index=False)
      .agg(
          Passengers=("Passengers", "sum"),
          Average_Delay=("Average_Delay_Minutes", "mean"),
          On_Time=("On_Time_Percentage", "mean")
      )
      .sort_values("Passengers", ascending=False)
)

fig3 = px.bar(
    mode_data,
    x="Transport_Mode",
    y="Passengers",
    color="Average_Delay",
    text_auto=".2s",
    title="3. Passenger Demand by Transport Mode",
    labels={
        "Transport_Mode": "Transport mode",
        "Passengers": "Total passengers",
        "Average_Delay": "Average delay (minutes)"
    },
    template="plotly_white",
    color_continuous_scale="RdYlGn_r"
)
fig3.show()
# ------------------------- 6. CHART 3 -----------------------------
# Transport mode comparison

mode_data = (
    df.groupby("Transport_Mode", as_index=False)
      .agg(
          Passengers=("Passengers", "sum"),
          Average_Delay=("Average_Delay_Minutes", "mean"),
          On_Time=("On_Time_Percentage", "mean")
      )
      .sort_values("Passengers", ascending=False)
)

fig3 = px.bar(
    mode_data,
    x="Transport_Mode",
    y="Passengers",
    color="Average_Delay",
    text_auto=".2s",
    title="3. Passenger Demand by Transport Mode",
    labels={
        "Transport_Mode": "Transport mode",
        "Passengers": "Total passengers",
        "Average_Delay": "Average delay (minutes)"
    },
    template="plotly_white",
    color_continuous_scale="RdYlGn_r"
)
fig3.show()


In [8]:
# ------------------------- 7. CHART 4 -----------------------------
# Relationship chart: delay versus punctuality

relationship_data = (
    df.groupby(["City", "Transport_Mode"], as_index=False)
      .agg(
          Average_Delay=("Average_Delay_Minutes", "mean"),
          On_Time=("On_Time_Percentage", "mean"),
          Passengers=("Passengers", "sum")
      )
)

fig4 = px.scatter(
    relationship_data,
    x="Average_Delay",
    y="On_Time",
    size="Passengers",
    color="Transport_Mode",
    hover_name="City",
    size_max=45,
    title="4. Relationship Between Average Delay and On-Time Performance",
    labels={
        "Average_Delay": "Average delay (minutes)",
        "On_Time": "On-time performance (%)",
        "Passengers": "Total passengers"
    },
    template="plotly_white"
)
fig4.show()



In [9]:
# ------------------------- 8. CHART 5 -----------------------------
# Distribution chart: delay histogram

fig5 = px.histogram(
    df.dropna(subset=["Average_Delay_Minutes"]),
    x="Average_Delay_Minutes",
    nbins=25,
    marginal="box",
    title="5. Distribution of Average Journey Delays",
    labels={"Average_Delay_Minutes": "Average delay (minutes)"},
    template="plotly_white"
)
fig5.show()



In [10]:
# ------------------------- 9. CHART 6 -----------------------------
# Customer rating box plot by transport mode

fig6 = px.box(
    df.dropna(subset=["Customer_Rating"]),
    x="Transport_Mode",
    y="Customer_Rating",
    color="Transport_Mode",
    points="outliers",
    title="6. Customer Rating Distribution by Transport Mode",
    labels={
        "Transport_Mode": "Transport mode",
        "Customer_Rating": "Customer rating (out of 5)"
    },
    template="plotly_white"
)
fig6.update_layout(showlegend=False)
fig6.show()



In [11]:
# ------------------------- 10. CHART 7 ----------------------------
# Financial comparison: revenue and operating cost by city

financial_data = (
    df.groupby("City", as_index=False)
      .agg(
          Ticket_Revenue=("Ticket_Revenue_EUR", "sum"),
          Operating_Cost=("Operating_Cost_EUR", "sum")
      )
)

financial_long = financial_data.melt(
    id_vars="City",
    value_vars=["Ticket_Revenue", "Operating_Cost"],
    var_name="Financial_Measure",
    value_name="Amount_EUR"
)

fig7 = px.bar(
    financial_long,
    x="City",
    y="Amount_EUR",
    color="Financial_Measure",
    barmode="group",
    title="7. Ticket Revenue Compared with Operating Costs",
    labels={
        "City": "City",
        "Amount_EUR": "Amount (€)",
        "Financial_Measure": "Financial measure"
    },
    template="plotly_white"
)
fig7.update_layout(xaxis_tickangle=-30)
fig7.show()



In [12]:
# ------------------------- 11. CHART 8 ----------------------------
# Correlation heatmap

correlation_columns = [
    "Passengers", "Trips", "Average_Delay_Minutes",
    "On_Time_Percentage", "Ticket_Revenue_EUR",
    "Operating_Cost_EUR", "CO2_Saved_KG",
    "Energy_Consumption_KWh", "Customer_Rating",
    "Accessibility_Complaints"
]

correlation_matrix = df[correlation_columns].corr()

fig8 = px.imshow(
    correlation_matrix,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="8. Correlation Between Mobility Performance Indicators",
    template="plotly_white"
)
fig8.show()



In [13]:
# ------------------------- 12. CHART 9 ----------------------------
# Accessibility complaints by city

accessibility_data = (
    df.groupby("City", as_index=False)["Accessibility_Complaints"]
      .sum()
      .sort_values("Accessibility_Complaints", ascending=False)
)

fig9 = px.bar(
    accessibility_data,
    x="City",
    y="Accessibility_Complaints",
    color="Accessibility_Complaints",
    title="9. Accessibility Complaints by City",
    labels={"Accessibility_Complaints": "Total complaints"},
    template="plotly_white",
    color_continuous_scale="Oranges"
)
fig9.update_layout(xaxis_tickangle=-30, coloraxis_showscale=False)
fig9.show()



In [14]:
# ------------------------- 13. CHART 10 ---------------------------
# Environmental impact: CO2 savings by transport mode

co2_data = (
    df.groupby("Transport_Mode", as_index=False)["CO2_Saved_KG"]
      .sum()
      .sort_values("CO2_Saved_KG", ascending=False)
)

fig10 = px.pie(
    co2_data,
    names="Transport_Mode",
    values="CO2_Saved_KG",
    hole=0.4,
    title="10. Share of CO2 Savings by Transport Mode",
    template="plotly_white"
)
fig10.show()


In [15]:
# ------------------------- 14. INTERACTIVE DASHBOARD --------------
# One combined Plotly dashboard with multiple interactive views.
# Hover, zoom, pan, legend selection and reset are available.

dashboard = make_subplots(
    rows=4,
    cols=2,
    subplot_titles=(
        "Monthly Passenger Demand",
        "Passenger Demand by City",
        "Passenger Demand by Transport Mode",
        "Delay vs On-Time Performance",
        "Distribution of Journey Delays",
        "Revenue vs Operating Cost by City",
        "Accessibility Complaints by City",
        "CO2 Savings by Transport Mode"
    ),
    specs=[
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "domain"}]
    ],
    vertical_spacing=0.10,
    horizontal_spacing=0.12
)

# Dashboard panel 1: monthly passenger demand
dashboard.add_trace(
    go.Scatter(
        x=monthly_data["Month"],
        y=monthly_data["Passengers"],
        mode="lines+markers",
        name="Passengers per month",
        hovertemplate="Month: %{x|%b %Y}<br>Passengers: %{y:,.0f}<extra></extra>"
    ),
    row=1, col=1
)

# Dashboard panel 2: city demand
dashboard.add_trace(
    go.Bar(
        x=city_data["City"],
        y=city_data["Passengers"],
        name="Passengers by city",
        hovertemplate="City: %{x}<br>Passengers: %{y:,.0f}<extra></extra>"
    ),
    row=1, col=2
)

# Dashboard panel 3: mode demand
dashboard.add_trace(
    go.Bar(
        x=mode_data["Transport_Mode"],
        y=mode_data["Passengers"],
        name="Passengers by mode",
        hovertemplate="Mode: %{x}<br>Passengers: %{y:,.0f}<extra></extra>"
    ),
    row=2, col=1
)

# Dashboard panel 4: relationship
for mode in relationship_data["Transport_Mode"].dropna().unique():
    part = relationship_data[
        relationship_data["Transport_Mode"] == mode
    ]
    dashboard.add_trace(
        go.Scatter(
            x=part["Average_Delay"],
            y=part["On_Time"],
            mode="markers",
            name=str(mode),
            marker=dict(size=10),
            customdata=part[["City", "Passengers"]].to_numpy(),
            hovertemplate=(
                "City: %{customdata[0]}<br>"
                "Average delay: %{x:.2f} minutes<br>"
                "On-time: %{y:.1f}%<br>"
                "Passengers: %{customdata[1]:,.0f}<extra></extra>"
            )
        ),
        row=2, col=2
    )

# Dashboard panel 5: delay distribution
dashboard.add_trace(
    go.Histogram(
        x=df["Average_Delay_Minutes"].dropna(),
        nbinsx=25,
        name="Delay distribution",
        showlegend=False
    ),
    row=3, col=1
)

# Dashboard panel 6: revenue and operating costs
for measure in ["Ticket_Revenue", "Operating_Cost"]:
    dashboard.add_trace(
        go.Bar(
            x=financial_data["City"],
            y=financial_data[measure],
            name=measure.replace("_", " "),
            hovertemplate="%{x}<br>€%{y:,.0f}<extra></extra>"
        ),
        row=3, col=2
    )

# Dashboard panel 7: accessibility complaints
dashboard.add_trace(
    go.Bar(
        x=accessibility_data["City"],
        y=accessibility_data["Accessibility_Complaints"],
        name="Accessibility complaints",
        showlegend=False
    ),
    row=4, col=1
)

# Dashboard panel 8: CO2 savings
dashboard.add_trace(
    go.Pie(
        labels=co2_data["Transport_Mode"],
        values=co2_data["CO2_Saved_KG"],
        hole=0.4,
        name="CO2 savings",
        showlegend=False,
        hovertemplate="%{label}<br>CO2 saved: %{value:,.1f} kg<br>%{percent}<extra></extra>"
    ),
    row=4, col=2
)

dashboard.update_layout(
    title={
        "text": "ECOMOVE MOBILITY - INTERACTIVE PERFORMANCE DASHBOARD",
        "x": 0.5,
        "xanchor": "center"
    },
    height=1500,
    template="plotly_white",
    barmode="group",
    hovermode="closest",
    legend_title_text="Measure"
)

dashboard.update_xaxes(title_text="Month", row=1, col=1)
dashboard.update_yaxes(title_text="Passengers", row=1, col=1)
dashboard.update_xaxes(title_text="City", row=1, col=2)
dashboard.update_yaxes(title_text="Passengers", row=1, col=2)
dashboard.update_xaxes(title_text="Transport mode", row=2, col=1)
dashboard.update_yaxes(title_text="Passengers", row=2, col=1)
dashboard.update_xaxes(title_text="Average delay (minutes)", row=2, col=2)
dashboard.update_yaxes(title_text="On-time performance (%)", row=2, col=2)
dashboard.update_xaxes(title_text="Average delay (minutes)", row=3, col=1)
dashboard.update_yaxes(title_text="Frequency", row=3, col=1)
dashboard.update_xaxes(title_text="City", row=3, col=2)
dashboard.update_yaxes(title_text="Amount (€)", row=3, col=2)
dashboard.update_xaxes(title_text="City", row=4, col=1)
dashboard.update_yaxes(title_text="Complaints", row=4, col=1)

dashboard.show()


In [16]:
# ------------------------- 15. KPI SUMMARY ------------------------
total_passengers = df["Passengers"].sum()
total_trips = df["Trips"].sum()
total_revenue = df["Ticket_Revenue_EUR"].sum()
total_cost = df["Operating_Cost_EUR"].sum()
total_co2_saved = df["CO2_Saved_KG"].sum()
average_delay = df["Average_Delay_Minutes"].mean()
average_on_time = df["On_Time_Percentage"].mean()
average_rating = df["Customer_Rating"].mean()
total_complaints = df["Accessibility_Complaints"].sum()

kpi_summary = pd.DataFrame({
    "Performance Indicator": [
        "Total passengers",
        "Total trips",
        "Total ticket revenue (€)",
        "Total operating cost (€)",
        "Net operating result (€)",
        "Total CO2 saved (kg)",
        "Average delay (minutes)",
        "Average on-time performance (%)",
        "Average customer rating",
        "Total accessibility complaints"
    ],
    "Value": [
        total_passengers,
        total_trips,
        round(total_revenue, 2),
        round(total_cost, 2),
        round(total_revenue - total_cost, 2),
        round(total_co2_saved, 2),
        round(average_delay, 2),
        round(average_on_time, 2),
        round(average_rating, 2),
        total_complaints
    ]
})

print("\nKEY PERFORMANCE INDICATORS")
display(kpi_summary)



KEY PERFORMANCE INDICATORS


,Performance Indicator,Value
0,Total passengers,4.090972e+07
1,Total trips,6.088620e+05
2,Total ticket revenue (€),1.116406e+08
3,Total operating cost (€),1.121100e+08
4,Net operating result (€),-4.694017e+05
5,Total CO2 saved (kg),4.481974e+06
6,Average delay (minutes),4.350000e+00
7,Average on-time performance (%),8.206000e+01
8,Average customer rating,4.010000e+00
9,Total accessibility complaints,6.266100e+04


In [17]:
# ------------------------- 16. EXPORT RESULTS ----------------------
# Save the dashboard and individual charts as interactive HTML files.
# These can be opened in a web browser and inserted/linked in your report.

output_folder = Path("EcoMove_Plotly_Outputs")
output_folder.mkdir(exist_ok=True)

figures = {
    "01_monthly_passenger_demand": fig1,
    "02_passenger_demand_by_city": fig2,
    "03_transport_mode_comparison": fig3,
    "04_delay_vs_punctuality": fig4,
    "05_delay_distribution": fig5,
    "06_customer_ratings": fig6,
    "07_revenue_vs_cost": fig7,
    "08_correlation_heatmap": fig8,
    "09_accessibility_complaints": fig9,
    "10_co2_savings": fig10,
    "11_interactive_dashboard": dashboard
}

for filename, figure in figures.items():
    figure.write_html(
        output_folder / f"{filename}.html",
        include_plotlyjs=True
    )

kpi_summary.to_csv(
    output_folder / "kpi_summary.csv",
    index=False
)
city_summary.to_csv(
    output_folder / "city_performance_summary.csv"
)

print(f"\nAll charts and summaries saved in: {output_folder.resolve()}")
print("Open 11_interactive_dashboard.html in a web browser to view the dashboard.")
print("Investigation completed successfully.")




All charts and summaries saved in: /content/EcoMove_Plotly_Outputs
Open 11_interactive_dashboard.html in a web browser to view the dashboard.
Investigation completed successfully.
